In [1]:
# Ячейка 1: скомпилировать (один раз или при изменении .cpp)
!g++ -O3 -shared -fPIC -std=c++17 mendrive_core.cpp -o libmendrive.so

In [2]:
from mendrive_ctypes import lib, MenDriveCpp

In [3]:
# Ячейка: перенесённая логика main() -- скан, FFT, сходимость, гистерезис, сравнение
import numpy as np
import matplotlib.pyplot as plt
import time, os

def shoelace_area(x, y):
    """Площадь замкнутой кривой (проверка петли гистерезиса)."""
    return 0.5*abs(np.sum(x*np.roll(y,-1) - np.roll(x,-1)*y))

def scan_resonance_cpp(N, freqs, ferrite_model='JA', n_sub=1, n_fp=1, n_newton=3, **kwargs):
    resp = []
    for w in freqs:
        sim = MenDriveCpp(N, ferrite_model=ferrite_model, n_sub=n_sub, n_fp=n_fp, n_newton=n_newton, **kwargs)
        res = sim.run(w, n_periods=3, record_from_period=1, amp=0.3, ramp_periods=1)
        a_resp = (res['Hn'].max()-res['Hn'].min())/2 if (len(res['Hn'])>0 and not res['blew_up']) else np.nan
        resp.append(a_resp)
    return np.array(resp)

def main_cpp(N=20, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
             excitation_mode='magnetic_right', sigma_e_left=3.0, sigma_m_leak=3.0,
             Ms=1.0, a_JA=0.3, alpha_JA=0.001, k_JA=0.15, c_JA=0.15,
             Ms_llg=0.3, gamma_llg=1.0, alpha_llg=0.1,
             n_sub=2, n_fp=2, n_newton=4,
             freqs_wide=None, freqs_fine_halfwidth=0.9, freqs_fine_step=0.1,
             n_periods_total=80, record_from_period=15, amp=1.0, ramp_periods=2.0,
             probe_idx=0, last_frac_force=0.5,
             make_plots=True, out_dir='./outputs', show_plots=False, verbose=True):
    """Полный порт main() на C++ ядро вместо чистого Python. См. docstring
    в предыдущем ответе для подробного описания шагов и возвращаемых ключей."""
    os.makedirs(out_dir, exist_ok=True)
    t_start = time.time()
    common_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                          Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                          Ms=Ms, a_JA=a_JA, alpha_JA=alpha_JA, k_JA=k_JA, c_JA=c_JA,
                          sigma_m_leak=sigma_m_leak, sigma_e_left=sigma_e_left,
                          excitation_mode=excitation_mode)
    plots = []

    if freqs_wide is None:
        freqs_wide = np.arange(1.0, 16.01, 0.5)
    resp_wide = scan_resonance_cpp(N, freqs_wide, ferrite_model=ferrite_model,
                                    n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    omega_coarse = freqs_wide[np.nanargmax(resp_wide)]
    if verbose:
        print(f"[main_cpp] Грубый скан: пик omega0~{omega_coarse:.2f}, {time.time()-t_start:.2f}с")

    lo = max(freqs_wide[0], omega_coarse - freqs_fine_halfwidth)
    hi = omega_coarse + freqs_fine_halfwidth
    freqs_fine = np.arange(lo, hi + 1e-9, freqs_fine_step)
    resp_fine = scan_resonance_cpp(N, freqs_fine, ferrite_model=ferrite_model,
                                    n_sub=1, n_fp=1, n_newton=3, **common_kwargs)
    omega_res = freqs_fine[np.nanargmax(resp_fine)] if not np.all(np.isnan(resp_fine)) else omega_coarse
    if verbose:
        print(f"[main_cpp] Уточнённый резонанс: omega0={omega_res:.3f}, {time.time()-t_start:.2f}с")

    if make_plots:
        plt.figure(figsize=(8,4))
        plt.plot(freqs_wide, resp_wide, 'o--', ms=3, alpha=0.5, label='грубый скан')
        plt.plot(freqs_fine, resp_fine, 'o-', ms=4, color='tab:blue', label='уточняющий скан')
        plt.axvline(omega_res, color='red', ls='--', label=f'omega0={omega_res:.2f}')
        plt.xlabel('omega0'); plt.ylabel('амплитуда отклика Hz')
        plt.title(f'{ferrite_model} (C++): скан резонанса')
        plt.legend(); plt.grid(True); plt.tight_layout()
        p = os.path.join(out_dir, f'scan_{ferrite_model}_cpp.png')
        plt.savefig(p, dpi=120);
        if show_plots:
            plt.show();
        plt.close(); plots.append(p)

    sim = MenDriveCpp(N, ferrite_model=ferrite_model, n_sub=n_sub, n_fp=n_fp,
                       n_newton=n_newton, **common_kwargs)
    res_long = sim.run(omega_res, n_periods=n_periods_total,
                        record_from_period=record_from_period, amp=amp,
                        ramp_periods=ramp_periods, probe_idx=probe_idx)
    if verbose:
        n_cov = len(res_long['t'])*res_long['dt']/res_long['T'] if len(res_long['t'])>0 else 0.0
        print(f"[main_cpp] Длинный прогон: {time.time()-t_start:.2f}с, точек={len(res_long['t'])}, "
              f"blew_up={res_long['blew_up']}, периодов записи~{n_cov:.1f}")

    res_llg_ref = None
    if ferrite_model == 'Hybrid':
        llg_kwargs = dict(bias_orientation=bias_orientation, H0_bias=H0_bias,
                           Ms_llg=Ms_llg, gamma_llg=gamma_llg, alpha_llg=alpha_llg,
                           sigma_e_left=sigma_e_left, excitation_mode=excitation_mode)
        sim_llg = MenDriveCpp(N, ferrite_model='LLG', **llg_kwargs)
        res_llg_ref = sim_llg.run(omega_res, n_periods=n_periods_total,
                                   record_from_period=record_from_period, amp=amp,
                                   ramp_periods=ramp_periods, probe_idx=probe_idx)

    t_h, dTxx_h, dt_h, T_h = res_long['t'], res_long['dTxx'], res_long['dt'], res_long['T']
    fft_freqs, fft_mag = np.array([]), np.array([])
    if len(dTxx_h) >= 8:
        Nfft = len(dTxx_h)
        window = np.hanning(Nfft)
        spec = np.fft.rfft(dTxx_h * window)
        fft_freqs = np.fft.rfftfreq(Nfft, d=dt_h) * 2*np.pi
        fft_mag = np.abs(spec)
        if make_plots:
            plt.figure(figsize=(9,4))
            plt.plot(fft_freqs, fft_mag, lw=1.0)
            plt.axvline(omega_res, color='red', ls='--', alpha=0.6, label=f'omega0={omega_res:.2f}')
            if bias_orientation in ('x','y','z') and ferrite_model in ('LLG','Hybrid'):
                plt.axvline(gamma_llg*H0_bias, color='green', ls='--', alpha=0.6,
                            label=f'gamma*H0={gamma_llg*H0_bias:.2f}')
            plt.xlim(0, min(fft_freqs.max(), 4*omega_res))
            plt.xlabel('omega'); plt.ylabel('|FFT(dTxx)|')
            plt.title(f'{ferrite_model} (C++): спектр dTxx')
            plt.legend(); plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, f'fft_dTxx_{ferrite_model}_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    convergence_ratio_cum = np.array([])
    spp_h = int(round(T_h/dt_h)) if len(t_h) > 0 else 0
    if spp_h > 0:
        n_rec_periods = len(t_h) // spp_h
        if n_rec_periods >= 1:
            pm_dTxx = np.array([dTxx_h[i*spp_h:(i+1)*spp_h].mean() for i in range(n_rec_periods)])
            pm_P = np.array([res_long['P'][i*spp_h:(i+1)*spp_h].mean() for i in range(n_rec_periods)])
            cum_F = np.cumsum(pm_dTxx) / np.arange(1, n_rec_periods+1)
            cum_P = np.cumsum(pm_P) / np.arange(1, n_rec_periods+1)
            convergence_ratio_cum = np.where(np.abs(cum_P) > 1e-30, cum_F/cum_P, np.nan)
            if make_plots:
                plt.figure(figsize=(9,4))
                plt.plot(np.arange(1, n_rec_periods+1), convergence_ratio_cum, 'o-', ms=3)
                plt.xlabel('периодов усреднено'); plt.ylabel('накопл. среднее dTxx/P')
                plt.title(f'{ferrite_model} (C++): сходимость force/power')
                plt.grid(True); plt.tight_layout()
                p = os.path.join(out_dir, f'convergence_{ferrite_model}_cpp.png')
                plt.savefig(p, dpi=120);
                if show_plots:
                    plt.show();
                plt.close(); plots.append(p)

    hysteresis_area = None
    if ferrite_model == 'Hybrid' and spp_h > 0 and len(res_long['HzJA']) >= spp_h:
        Hloop = res_long['HzJA'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['MzJA'][-spp_h:]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(8,8))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:green', marker='o', ms=2)
            plt.xlabel('H (Э)'); plt.ylabel('B (Гс)')
            plt.title(f'Hybrid: внутренняя петля JA (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, 'hysteresis_Hybrid_internalJA_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)
    elif ferrite_model in ['JA', 'Preisach'] and spp_h > 0:
        Hloop = res_long['Hn'][-spp_h:]
        Bloop = Hloop + 4*np.pi*res_long['Mn'][-spp_h:, 1]
        hysteresis_area = shoelace_area(Hloop, Bloop)
        if make_plots:
            plt.figure(figsize=(8,8))
            plt.plot(Hloop, Bloop, lw=1.5, color='tab:purple', marker='o', ms=2)
            plt.xlabel('H (Э)'); plt.ylabel('B (Гс)')
            plt.title(f'{ferrite_model}: петля гистерезиса (площадь={hysteresis_area:.3e})')
            plt.grid(True); plt.tight_layout()
            p = os.path.join(out_dir, f'hysteresis_{ferrite_model}_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    if ferrite_model == 'Hybrid' and res_llg_ref is not None and spp_h > 0:
        n_traj = min(3*spp_h, len(res_long['Mn']), len(res_llg_ref['Mn']))
        if n_traj > 0:
            fig, axes = plt.subplots(1, 2, figsize=(11,5))
            axes[0].plot(res_long['Mn'][-n_traj:,1], res_long['Mn'][-n_traj:,2], lw=0.8, color='tab:green')
            axes[0].set_title('Hybrid: траектория M'); axes[0].set_aspect('equal'); axes[0].grid(True)
            axes[1].plot(res_llg_ref['Mn'][-n_traj:,1], res_llg_ref['Mn'][-n_traj:,2], lw=0.8, color='tab:orange')
            axes[1].set_title('Чистый LLG: траектория M'); axes[1].set_aspect('equal'); axes[1].grid(True)
            plt.tight_layout()
            p = os.path.join(out_dir, 'M_trajectory_Hybrid_vs_LLG_cpp.png')
            plt.savefig(p, dpi=120);
            if show_plots:
                plt.show();
            plt.close(); plots.append(p)

    force_per_power_code, force_per_kW = sim.force_per_power(res_long, last_frac=last_frac_force)
    if verbose:
        print(f"[main_cpp] FORCE/POWER: {force_per_power_code:.4e} (код.ед.), {force_per_kW:.4e} Н/кВт")

    P_avg, Pdiss_avg, rel_dif = sim.check_energy_balance(res_long, last_frac=last_frac_force)
    if verbose:
        print(f"[main_cpp] check_energy_balance: P_avg = {P_avg:.4e}, Pdiss_avg={Pdiss_avg:.4e} rel_dif={rel_dif}")
        print(f"[main_cpp] Итого времени: {time.time()-t_start:.2f}с")

    return dict(omega_res=omega_res, freqs_wide=freqs_wide, resp_wide=resp_wide,
                freqs_fine=freqs_fine, resp_fine=resp_fine,
                res_long=res_long, res_long_llg_ref=res_llg_ref,
                hysteresis_area=hysteresis_area, fft_freqs=fft_freqs, fft_mag=fft_mag,
                convergence_ratio_cum=convergence_ratio_cum,
                force_per_power_code=force_per_power_code, force_per_kW=force_per_kW, plots=plots)

In [4]:
N = 160

In [5]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 1)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.00с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.33с
[main_cpp] Длинный прогон: 598.12с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: -1.5138e-08 (код.ед.), -1.5138e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 7.3452e+00, Pdiss_avg=3.6727e-03 rel_dif=0.9994999893528365
[main_cpp] Итого времени: 600.38с
omega_res = 14.799999999999997
force_per_kW = -0.0015138332680123174
hysteresis_area = 1.1102230246251565e-15


In [6]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 2)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.02с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.35с
[main_cpp] Длинный прогон: 602.00с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: -1.5138e-08 (код.ед.), -1.5138e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.9381e+01, Pdiss_avg=1.4691e-02 rel_dif=0.9994999893528365
[main_cpp] Итого времени: 604.25с
omega_res = 14.799999999999997
force_per_kW = -0.0015138332680123183
hysteresis_area = 1.9984014443252818e-15


In [7]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 3)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 604.42с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: -1.5138e-08 (код.ед.), -1.5138e-03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 6.6107e+01, Pdiss_avg=3.3054e-02 rel_dif=0.9994999893528366
[main_cpp] Итого времени: 606.66с
omega_res = 14.799999999999997
force_per_kW = -0.0015138332680123168
hysteresis_area = 1.5543122344752192e-15


In [8]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 4)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 606.42с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 3.8702e-10 (код.ед.), 3.8702e-05 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.1781e+02, Pdiss_avg=5.9465e-02 rel_dif=0.9994952310346361
[main_cpp] Итого времени: 608.69с
omega_res = 14.799999999999997
force_per_kW = 3.87018712195201e-05
hysteresis_area = 0.3038570383417014


In [9]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 5)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 626.45с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 1.3512e-02 (код.ед.), 1.3512e+03 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.4551e+02, Pdiss_avg=8.3875e+01 rel_dif=0.42359256207959434
[main_cpp] Итого времени: 628.76с
omega_res = 14.799999999999997
force_per_kW = 1351.2425663553668
hysteresis_area = 3135.0700667822657


In [10]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 6)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.03с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.36с
[main_cpp] Длинный прогон: 627.49с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 9.0870e-03 (код.ед.), 9.0870e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.1654e+02, Pdiss_avg=8.3748e+01 rel_dif=0.6132409256970521
[main_cpp] Итого времени: 629.77с
omega_res = 14.799999999999997
force_per_kW = 908.7007755305573
hysteresis_area = 3921.6046506031844


In [11]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 7)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 627.93с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 6.6048e-03 (код.ед.), 6.6048e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.9841e+02, Pdiss_avg=8.3832e+01 rel_dif=0.7190707524088449
[main_cpp] Итого времени: 630.21с
omega_res = 14.799999999999997
force_per_kW = 660.4786587338017
hysteresis_area = 2925.9593979061356


In [12]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 8)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 628.11с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.9605e-03 (код.ед.), 4.9605e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.9402e+02, Pdiss_avg=8.3730e+01 rel_dif=0.7875009430718581
[main_cpp] Итого времени: 630.38с
omega_res = 14.799999999999997
force_per_kW = 496.04662114361315
hysteresis_area = 2685.9114393514105


In [13]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 9)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 627.90с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.0335e-03 (код.ед.), 4.0335e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 4.8801e+02, Pdiss_avg=8.3773e+01 rel_dif=0.828336401846649
[main_cpp] Итого времени: 630.26с
omega_res = 14.799999999999997
force_per_kW = 403.3457962068422
hysteresis_area = 3056.6406287178065


In [14]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='magnetic_right',
                   n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                   n_periods_total=8000, record_from_period=1000,  amp = 10)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~15.00, 16.01с
[main_cpp] Уточнённый резонанс: omega0=14.800, 19.34с
[main_cpp] Длинный прогон: 627.88с, точек=5943553, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 3.2344e-03 (код.ед.), 3.2344e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 6.0619e+02, Pdiss_avg=8.3814e+01 rel_dif=0.8617360750842863
[main_cpp] Итого времени: 630.14с
omega_res = 14.799999999999997
force_per_kW = 323.4363277108653
hysteresis_area = 3169.683880667585


In [15]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=1)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.69с
[main_cpp] Длинный прогон: 653.00с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 6.8775e+00, Pdiss_avg=3.4712e-03 rel_dif=0.9994952795535738
[main_cpp] Итого времени: 655.48с
omega_res = 13.499999999999996
force_per_kW = 0.00045474067685358655
hysteresis_area = 6.661338147750939e-16


In [16]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=2)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.97с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.67с
[main_cpp] Длинный прогон: 655.77с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.7510e+01, Pdiss_avg=1.3885e-02 rel_dif=0.9994952795535738
[main_cpp] Итого времени: 658.24с
omega_res = 13.499999999999996
force_per_kW = 0.0004547406768535856
hysteresis_area = 8.881784197001252e-16


In [17]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=3)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.98с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.69с
[main_cpp] Длинный прогон: 658.12с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 6.1898e+01, Pdiss_avg=3.1241e-02 rel_dif=0.9994952795535738
[main_cpp] Итого времени: 660.67с
omega_res = 13.499999999999996
force_per_kW = 0.0004547406768535904
hysteresis_area = 8.881784197001252e-16


In [18]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=4)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.69с
[main_cpp] Длинный прогон: 658.17с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.1004e+02, Pdiss_avg=5.5540e-02 rel_dif=0.9994952795535738
[main_cpp] Итого времени: 660.63с
omega_res = 13.499999999999996
force_per_kW = 0.00045474067685358704
hysteresis_area = 1.7763568394002505e-15


In [19]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=5)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.70с
[main_cpp] Длинный прогон: 659.01с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 1.7194e+02, Pdiss_avg=8.6781e-02 rel_dif=0.9994952795535736
[main_cpp] Итого времени: 661.47с
omega_res = 13.499999999999996
force_per_kW = 0.00045474067685359447
hysteresis_area = 3.1086244689504383e-15


In [20]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=6)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.69с
[main_cpp] Длинный прогон: 660.57с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.5474e-09 (код.ед.), 4.5474e-04 Н/кВт
[main_cpp] check_energy_balance: P_avg = 2.4759e+02, Pdiss_avg=1.2496e-01 rel_dif=0.9994952795535738
[main_cpp] Итого времени: 663.05с
omega_res = 13.499999999999996
force_per_kW = 0.0004547406768535914
hysteresis_area = 4.440892098500626e-16


In [21]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=7)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.97с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.67с
[main_cpp] Длинный прогон: 687.04с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 5.7814e-03 (код.ед.), 5.7814e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 3.4093e+02, Pdiss_avg=8.3944e+01 rel_dif=0.7537798146036002
[main_cpp] Итого времени: 689.62с
omega_res = 13.499999999999996
force_per_kW = 578.1409751518233
hysteresis_area = 3845.5504097233274


In [ ]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=8)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.70с
[main_cpp] Длинный прогон: 687.05с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 4.4236e-03 (код.ед.), 4.4236e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 4.4701e+02, Pdiss_avg=8.3990e+01 rel_dif=0.8121068583482225
[main_cpp] Итого времени: 689.55с
omega_res = 13.499999999999996
force_per_kW = 442.36335282459186
hysteresis_area = 3311.855025561983


In [ ]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=9)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.98с
[main_cpp] Уточнённый резонанс: omega0=13.500, 19.68с
[main_cpp] Длинный прогон: 686.70с, точек=6515896, blew_up=False, периодов записи~7000.0
[main_cpp] FORCE/POWER: 3.4914e-03 (код.ед.), 3.4914e+02 Н/кВт
[main_cpp] check_energy_balance: P_avg = 5.6334e+02, Pdiss_avg=8.4022e+01 rel_dif=0.8508503629604922
[main_cpp] Итого времени: 689.20с
omega_res = 13.499999999999996
force_per_kW = 349.14251742295943
hysteresis_area = 3541.398235496225


In [ ]:
result = main_cpp(N=N, ferrite_model='Preisach', bias_orientation='x', H0_bias=4.0,
                  excitation_mode='electric_left',
                  n_sub=2, n_fp=2, n_newton=4, sigma_m_leak=3.0, sigma_e_left=3.0,
                  n_periods_total=8000, record_from_period=1000, amp=10)
print("omega_res =", result['omega_res'])
print("force_per_kW =", result['force_per_kW'])
print("hysteresis_area =", result['hysteresis_area'])

[main_cpp] Грубый скан: пик omega0~13.50, 15.99с
